# Robustness of Accessibility
## Notebook 2/4 - create disrupted graph based on normal graph and hazard data

In [ ]:
import geopandas as gpd
from shapely.ops import unary_union
import osmnx as ox

In [ ]:
from pathlib import Path

from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    RoaNotebookConfig,
    get_roa_cache_path,
    get_roa_hazard_data_path,
    get_roa_inputs_path,
)

event = RoaNotebookConfig.event

inputs_dir: Path = get_roa_inputs_path()
cache_dir: Path = get_roa_cache_path()
hazard_data_path: Path = get_roa_hazard_data_path(event=event)

In [ ]:
undirected_graph = RoaNotebookConfig.undirected_graph
place_name = RoaNotebookConfig.place_name
show_all_plots = False

In [ ]:
# Import shapes of flooded areas
hazard_area = gpd.read_file(hazard_data_path)
hazrd_area_geoms = hazard_area.geometry

# Combine all shapes of the flooded are into one to make it easier to work with
hazard_area_multipolygon_df = gpd.GeoSeries(unary_union(hazrd_area_geoms))
hazard_area_multipolygon = hazard_area_multipolygon_df[0]

### Load default graph from file and extract nodes and edges again

In [ ]:
road_network = ox.load_graphml(cache_dir / f"network/drive_graph_{place_name}.graphml")

In [ ]:
if undirected_graph:
    road_network = road_network.to_undirected()

In [ ]:
## 1. convert the graph to GeoDataFrame
road_network_nodes, road_network_edges = ox.graph_to_gdfs(road_network)

In [ ]:
print(f"number of nodes in all graph: {len(road_network)}")

#### Calculate the intersection of the flooded areas with the graph

In [ ]:
# identify the flooded areas / edges
mask_affected_edges = road_network_edges.intersects(hazard_area_multipolygon)  # (within(
affected_edges = road_network_edges[mask_affected_edges]

In [ ]:
# identify the flooded nodes
mask_affected_nodes = road_network_nodes.intersects(hazard_area_multipolygon)  # (within(
affected_nodes = road_network_nodes[mask_affected_nodes]

#### Identify bridges to exclude them from the flooded areas

In [ ]:
bridges = road_network_edges[road_network_edges["bridge"].notnull()]
bridges.plot()

In [ ]:
bridge_ids = set(bridges.index.values)
# excludes edges that are bridges from the affected_edges
mask_affected_edges_without_bridges = ~affected_edges.index.isin(bridge_ids)
affected_edges_without_bridges = affected_edges.loc[mask_affected_edges_without_bridges]

#### Validation of the flooded areas

In [ ]:
# affected_edges = gdf_edges_drive_graph_trier_default[mask_flooded_edges_trier]
affected_edges_without_bridges_ids = set(affected_edges_without_bridges.index.values)
# un_affected_edges = gdf_edges_drive_graph_trier_default[~mask_flooded_edges_trier]
un_affected_edges_including_bridges = road_network_edges.loc[
    ~road_network_edges.index.isin(affected_edges_without_bridges_ids)
]
if len(road_network_edges) == len(affected_edges_without_bridges) + len(un_affected_edges_including_bridges):
    print(
        f"Edges are split correctly: Affected edges without bridges = {len(affected_edges_without_bridges)} unaffected edges including bridges = {len(un_affected_edges_including_bridges)} total = {len(road_network_edges)}"
    )
else:
    raise RuntimeError("edges are not correctly split")

In [ ]:
## create network from gdfs i.e. nodes and edges
# uses all nodes now to avoid any problems with graph / route calculation
road_network_remaining = ox.graph_from_gdfs(road_network_nodes, un_affected_edges_including_bridges)
# road_network_removed can be used for visualization - not relevant for the calculation
road_network_removed = ox.graph_from_gdfs(road_network_nodes, affected_edges_without_bridges)

In [ ]:
if undirected_graph:
    road_network_remaining = road_network_remaining.to_undirected()

In [ ]:
ox.save_graphml(
    road_network_removed,
    filepath=cache_dir / f"network/drive_graph_removed{place_name}.graphml",
)
ox.save_graphml(
    road_network_remaining,
    filepath=cache_dir / f"network/drive_graph_remaining_{place_name}.graphml",
)